In [2]:
import json
import os
import matplotlib.pyplot as plt
import networkx as nx
import json
import re
import json
from graphviz import Digraph
import matplotlib.pyplot as plt
from PIL import Image

def read_txt_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    return content

def save_json(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4)

def read_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

In [24]:
def parse_criteria(criteria_lines):
    criteria_tree = []
    for line in criteria_lines:
        line = line.strip()
        if "[AND]" in line or "[OR]" in line:
            parts = re.split(r'\s*\[\s*(AND|OR)\s*\]\s*', line)
            for i in range(len(parts)):
                if parts[i].strip():
                    criteria_tree.append(parts[i].strip())
                    if i < len(parts) - 1:
                        criteria_tree.append(parts[i + 1].strip())
        else:
            criteria_tree.append(line)
    return criteria_tree

def build_tree_from_text(criteria_tree, conjunction="AND"):
    if not criteria_tree:
        return None
    if len(criteria_tree) == 1:
        return {"raw_text": criteria_tree[0]}
    if len(criteria_tree) == 2:
        return {conjunction: {"left": {"raw_text": criteria_tree[0]}, "right": {"raw_text": criteria_tree[1]}}}

    mid = len(criteria_tree) // 2
    left_tree = build_tree_from_text(criteria_tree[:mid], conjunction)
    right_tree = build_tree_from_text(criteria_tree[mid:], conjunction)
    return {conjunction: {"left": left_tree, "right": right_tree}}

def build_tree_with_conjunctions(criteria_tree):
    if not criteria_tree:
        return None

    def split_criteria(criteria_list):
        result = []
        temp_list = []
        for criteria in criteria_list:
            if criteria.upper() in ["AND", "OR"]:
                if temp_list:
                    result.append(temp_list)
                    temp_list = []
                result.append(criteria.upper())
            else:
                temp_list.append(criteria)
        if temp_list:
            result.append(temp_list)
        return result

    def recursive_build(criteria_list):
        if not criteria_list:
            return None
        if len(criteria_list) == 1 and isinstance(criteria_list[0], list):
            return build_tree_from_text(criteria_list[0], "AND")
        if len(criteria_list) == 1:
            return {"raw_text": criteria_list[0]}

        if len(criteria_list) == 3:
            left = {"raw_text": criteria_list[0][0]} if isinstance(criteria_list[0], list) else {"raw_text": criteria_list[0]}
            operator = criteria_list[1]
            right = {"raw_text": criteria_list[2][0]} if isinstance(criteria_list[2], list) else {"raw_text": criteria_list[2]}
            return {operator: {"left": left, "right": right}}

        operator = criteria_list[-2]
        left = recursive_build(criteria_list[:-2])
        right = {"raw_text": criteria_list[-1][0]} if isinstance(criteria_list[-1], list) else {"raw_text": criteria_list[-1]}

        return {operator: {"left": left, "right": right}}

    split_list = split_criteria(criteria_tree)
    return recursive_build(split_list)

def create_tree_structure(text):
    lines = text.split('\n')
    inclusion_criteria = []
    exclusion_criteria = []

    current_list = None
    for line in lines:
        line = line.strip()
        if line.startswith("Inclusion Criteria"):
            current_list = inclusion_criteria
        elif line.startswith("Exclusion Criteria"):
            current_list = exclusion_criteria
        elif line.startswith('-') or re.match(r'^\d+\.', line):
            if current_list is not None:
                current_list.append(line[1:].strip() if line.startswith('-') else line.split('. ', 1)[1])

    inclusion_tree = build_tree_with_conjunctions(parse_criteria(inclusion_criteria))
    exclusion_tree = build_tree_with_conjunctions(parse_criteria(exclusion_criteria))

    return {"AND": {"left": inclusion_tree, "right": {"NOT OR": exclusion_tree}}}

def save_json(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4)

# Beispielaufruf
def read_txt_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    return content


In [25]:
file_path = 'data_parsed_1/NCT03860012.txt'
output_path = 'data_parsed_2/NCT03860012.json'


In [26]:
text = read_txt_file(file_path)
tree_structure = create_tree_structure(text)
save_json(tree_structure, output_path)

In [27]:
def parse_logic_to_tree(logic, graph, parent=None, node_id=0):
    if 'raw_text' in logic:
        node_label = logic['raw_text']
        graph.node(str(node_id), label=node_label, shape='box')
        if parent is not None:
            graph.edge(parent, str(node_id))
        return node_id

    for key in logic:
        if key in ('AND', 'OR', 'NOT'):
            operator = key
            node_label = operator
            current_node_id = node_id
            color = 'lightblue' if operator == 'AND' else 'lightgreen' if operator == 'OR' else 'lightcoral'
            graph.node(str(current_node_id), label=node_label, color=color, fontcolor='black', style='filled', fillcolor=color)
            if parent is not None:
                graph.edge(parent, str(current_node_id))

            operands = logic[key]
            node_id += 1
            if 'left' in operands:
                left_id = parse_logic_to_tree(operands['left'], graph, str(current_node_id), node_id)
                node_id = left_id + 1
            if 'right' in operands:
                right_id = parse_logic_to_tree(operands['right'], graph, str(current_node_id), node_id)
                node_id = right_id + 1
            if operator == 'NOT':
                node_id += 1
    return node_id

def visualize_logic_tree(logic):
    graph = Digraph(format='png')
    parse_logic_to_tree(logic, graph)
    return graph

def plot_and_save_graph(logic, graph_title, output_filename):
    graph = visualize_logic_tree(logic)
    graph.render(filename='temp', view=False, cleanup=False)
    image = Image.open('temp.png')
    plt.figure(figsize=(20, 15))
    plt.imshow(image)
    plt.axis('off')
    plt.title(graph_title)
    plt.savefig(output_filename)
    plt.show()
file_path = 'data_parsed_2/NCT03860012.json'  # Pfad zur JSON-Datei

tree = read_json(file_path)
logic_tree = read_json(file_path)

# Baumstruktur visualisieren und speichern
plot_and_save_graph(logic_tree, "Tree Structure Plot", "NCT03860012_tree_plot.png")